In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from pbase.analysis.tools.all_positions import MisoApTools


aptools = MisoApTools()

# Production Example: Training on Historical Data (2024-11 to 2025-09)
# Testing on Future Period (2025-11)

This section shows the complete workflow with real date ranges.

In [2]:
# ============================================================================
# STEP 1: Define Date Ranges and Parameters
# ============================================================================

print("=" * 80)
print("SHADOW PRICE PREDICTION: PRODUCTION WORKFLOW")
print("=" * 80)

# Test period: 2025-11 (ALL outage dates in this month)
test_auction_month = pd.Timestamp('2025-10')
test_market_month = pd.Timestamp('2025-10')

train_end = test_auction_month - pd.offsets.MonthBegin(1)
train_start = train_end - pd.offsets.MonthBegin(12)

# Fixed parameters
period_type = 'f0'
class_type = 'onpeak'
market_round = 1

print(f"\n[STEP 1] Date Ranges Defined")
print(f"  Training Period: {train_start.strftime('%Y-%m')} to {train_end.strftime('%Y-%m')}")
print(f"  Test Period: {test_auction_month.strftime('%Y-%m')} (ALL outage dates)")
print(f"  Class Type: {class_type}")
print(f"  Period Type: {period_type}")
print(f"\n  Note: Will predict for all available outage dates in the test month")

SHADOW PRICE PREDICTION: PRODUCTION WORKFLOW

[STEP 1] Date Ranges Defined
  Training Period: 2024-09 to 2025-09
  Test Period: 2025-10 (ALL outage dates)
  Class Type: onpeak
  Period Type: f0

  Note: Will predict for all available outage dates in the test month


In [3]:
# ============================================================================
# STEP 2: Helper Function to Load Data for a Given Month/Outage Date
# ============================================================================

def load_training_data_for_outage(auction_month, market_month, outage_date, 
                                   period_type='f0', class_type='onpeak', market_round=1):
    """
    Load training data (score features + shadow price labels) for a single outage date.
    
    Returns:
    --------
    DataFrame with columns: score, label, branch_name
    Index: (constraint_id, flow_direction)
    """
    try:
        # Build paths
        score_path_str = f'/opt/temp/tmp/pw_data/spice6/prod_f0p_model_miso/density/auction_month={{auction_month}}/market_month={{market_month}}/market_round={market_round}/outage_date={{outage_date}}'
        cons_path_str = f'/opt/temp/tmp/pw_data/spice6/prod_f0p_model_miso/constraint_info/auction_month={{auction_month}}/market_round={market_round}/period_type={{period_type}}/class_type={{class_type}}'
        
        score_path = Path(score_path_str.format(
            auction_month=auction_month.strftime('%Y-%m'),
            market_month=market_month.strftime('%Y-%m'),
            outage_date=outage_date.strftime('%Y-%m-%d')
        ))
        
        cons_path = Path(cons_path_str.format(
            auction_month=auction_month.strftime('%Y-%m'),
            period_type=period_type,
            class_type=class_type
        ))
        
        # Check if paths exist
        if not score_path.exists():
            return None
        
        # Load score data
        score_df = pd.read_parquet(score_path / 'score.parquet')
        
        # Load shadow prices for the outage period
        outage_st = outage_date
        outage_et = outage_date + pd.Timedelta(days=4)
        
        da_label = aptools.tools.get_da_shadow_by_peaktype(
            market_month, 
            market_month + pd.offsets.MonthBegin(1), 
            class_type
        )
        da_label.index = da_label.index.tz_localize(None)
        
        # Aggregate shadow prices for this outage period
        da_label_chunk = da_label.loc[
            (da_label.index >= outage_st) & (da_label.index < outage_et)
        ].groupby('constraint_id')['shadow_price'].sum()
        
        # Load constraint info
        if cons_path.exists():
            cons = pd.read_parquet(cons_path / 'constraints.parquet')
            spice_map = cons[cons['convention'] != 999].dropna(subset=['branch_name']).groupby('constraint_id')['branch_name'].apply(','.join)
            
            # Map to branch names
            score_df['branch_name'] = score_df.index.get_level_values('constraint_id').map(spice_map)
            da_label_chunk.index = da_label_chunk.index.map(spice_map)
            da_label_chunk = da_label_chunk.groupby(level=0).sum().abs()
        else:
            score_df['branch_name'] = score_df.index.get_level_values('constraint_id')
            da_label_chunk = da_label_chunk.abs()
        
        # Assign labels
        score_df['label'] = score_df['branch_name'].map(da_label_chunk).fillna(0)
        
        # Keep only one row per constraint_id (highest score)
        score_df = score_df.sort_values('score', ascending=False).reset_index().groupby('constraint_id').first().reset_index().set_index(['constraint_id', 'flow_direction'])
        
        # Add metadata
        score_df['auction_month'] = auction_month
        score_df['market_month'] = market_month
        score_df['outage_date'] = outage_date
        
        return score_df
        
    except Exception as e:
        print(f"  ⚠️  Error loading data for {outage_date.strftime('%Y-%m-%d')}: {str(e)}")
        return None

print("[STEP 2] Helper Function Defined")
print("  ✓ load_training_data_for_outage() ready to use")

[STEP 2] Helper Function Defined
  ✓ load_training_data_for_outage() ready to use


In [4]:
# ============================================================================
# STEP 3: Load Training Data (2024-11 to 2025-09)
# ============================================================================

print("\n[STEP 3] Loading Training Data")
print("-" * 80)

# Generate list of auction months in training period
training_months = pd.date_range(train_start, train_end, freq='MS', inclusive='left')

print(f"Training months: {len(training_months)} months")
print(f"  From: {training_months[0].strftime('%Y-%m')}")
print(f"  To:   {training_months[-1].strftime('%Y-%m')}")

# Collect all training data
training_data_list = []

for auction_month in training_months:
    market_month = auction_month  # Assuming auction_month == market_month
    
    print(f"\nProcessing {auction_month.strftime('%Y-%m')}...")
    
    # For each month, we need to find all available outage dates
    # Outage dates are typically every 3-4 days throughout the month
    # For simplicity, we'll sample common outage dates: 1st, 5th, 9th, 13th, 17th, 21st, 25th
    
    month_data_count = 0
    for outage_date in pd.date_range(market_month, market_month + pd.offsets.MonthBegin(1), freq='3D', inclusive='left'):
        try:
            # Skip if outage_date is beyond train_end
            if outage_date > train_end:
                continue
            
            # Load data for this outage date
            data = load_training_data_for_outage(
                auction_month=auction_month,
                market_month=market_month,
                outage_date=outage_date,
                period_type=period_type,
                class_type=class_type,
                market_round=market_round
            )
            
            if data is not None and len(data) > 0:
                training_data_list.append(data)
                month_data_count += 1
                print(f"  ✓ {outage_date.strftime('%Y-%m-%d')}: {len(data):,} constraints")
        except:
            # Silently skip dates that don't exist or have errors
            pass
    
    if month_data_count == 0:
        print(f"  ⚠️  No data found for {auction_month.strftime('%Y-%m')}")

# Combine all training data
if len(training_data_list) > 0:
    print(f"\n" + "=" * 80)
    print(f"Combining {len(training_data_list)} datasets...")
    train_data_combined = pd.concat(training_data_list, axis=0)
    
    print(f"\n✓ Training Data Loaded Successfully")
    print(f"  Total samples: {len(train_data_combined):,}")
    print(f"  Date range: {train_data_combined['outage_date'].min().strftime('%Y-%m-%d')} to {train_data_combined['outage_date'].max().strftime('%Y-%m-%d')}")
    print(f"  Unique constraints: {train_data_combined.index.get_level_values('constraint_id').nunique():,}")
    
    # Check class distribution
    binding_count = (train_data_combined['label'] > 0).sum()
    non_binding_count = len(train_data_combined) - binding_count
    print(f"\n  Class Distribution:")
    print(f"    Binding: {binding_count:,} ({binding_count/len(train_data_combined)*100:.2f}%)")
    print(f"    Non-binding: {non_binding_count:,} ({non_binding_count/len(train_data_combined)*100:.2f}%)")
else:
    print("\n⚠️  ERROR: No training data loaded!")
    train_data_combined = None


[STEP 3] Loading Training Data
--------------------------------------------------------------------------------
Training months: 12 months
  From: 2024-09
  To:   2025-08

Processing 2024-09...
  ✓ 2024-09-01: 12,636 constraints
  ✓ 2024-09-04: 12,533 constraints
  ✓ 2024-09-07: 12,478 constraints
  ✓ 2024-09-10: 12,408 constraints
  ✓ 2024-09-13: 12,499 constraints
  ✓ 2024-09-16: 12,292 constraints
  ✓ 2024-09-19: 12,384 constraints
  ✓ 2024-09-22: 12,458 constraints
  ✓ 2024-09-25: 12,446 constraints
  ✓ 2024-09-28: 12,403 constraints

Processing 2024-10...
  ✓ 2024-10-01: 12,177 constraints
  ✓ 2024-10-04: 12,298 constraints
  ✓ 2024-10-07: 12,203 constraints
  ✓ 2024-10-10: 12,212 constraints
  ✓ 2024-10-13: 12,251 constraints
  ✓ 2024-10-16: 12,196 constraints
  ✓ 2024-10-19: 12,255 constraints
  ✓ 2024-10-22: 12,197 constraints
  ✓ 2024-10-25: 12,293 constraints
  ✓ 2024-10-28: 12,257 constraints
  ✓ 2024-10-31: 12,308 constraints

Processing 2024-11...
  ✓ 2024-11-01: 12,335 c

In [5]:
# ============================================================================
# STEP 4: Load Test Data (ALL Outage Dates in Test Month)
# ============================================================================

print("\n[STEP 4] Loading Test Data (All Outage Dates)")
print("-" * 80)

print(f"Test period: {test_auction_month.strftime('%Y-%m')}")

# Collect all test data for all outage dates in the test month
test_data_list = []

# Generate outage dates for the entire test month (every 3 days)
test_outage_dates = pd.date_range(
    test_market_month, 
    test_market_month + pd.offsets.MonthBegin(1), 
    freq='3D', 
    inclusive='left'
)

print(f"  Scanning for outage dates in {test_market_month.strftime('%Y-%m')}...")

loaded_outage_count = 0
for outage_date in test_outage_dates:
    try:
        # Load data for this outage date
        data = load_training_data_for_outage(
            auction_month=test_auction_month,
            market_month=test_market_month,
            outage_date=outage_date,
            period_type=period_type,
            class_type=class_type,
            market_round=market_round
        )
        
        if data is not None and len(data) > 0:
            test_data_list.append(data)
            loaded_outage_count += 1
            print(f"  ✓ {outage_date.strftime('%Y-%m-%d')}: {len(data):,} constraints")
    except:
        # Silently skip dates that don't exist or have errors
        pass

# Combine all test data
if len(test_data_list) > 0:
    print(f"\n" + "=" * 80)
    print(f"Combining {len(test_data_list)} test datasets...")
    test_data = pd.concat(test_data_list, axis=0)
    
    print(f"\n✓ Test Data Loaded Successfully")
    print(f"  Total samples: {len(test_data):,}")
    print(f"  Outage dates loaded: {loaded_outage_count}")
    print(f"  Date range: {test_data['outage_date'].min().strftime('%Y-%m-%d')} to {test_data['outage_date'].max().strftime('%Y-%m-%d')}")
    print(f"  Unique constraints: {test_data.index.get_level_values('constraint_id').nunique():,}")
    
    # Check class distribution
    binding_count_test = (test_data['label'] > 0).sum()
    non_binding_count_test = len(test_data) - binding_count_test
    print(f"\n  Class Distribution:")
    print(f"    Binding: {binding_count_test:,} ({binding_count_test/len(test_data)*100:.2f}%)")
    print(f"    Non-binding: {non_binding_count_test:,} ({non_binding_count_test/len(test_data)*100:.2f}%)")
    
    # Show breakdown by outage date
    outage_counts = test_data.groupby('outage_date').size()
    print(f"\n  Samples per outage date:")
    for outage_dt, count in outage_counts.items():
        binding_count_per_date = (test_data[test_data['outage_date'] == outage_dt]['label'] > 0).sum()
        print(f"    {outage_dt.strftime('%Y-%m-%d')}: {count:,} samples ({binding_count_per_date:,} binding)")
else:
    print("\n⚠️  ERROR: No test data loaded!")
    test_data = None


[STEP 4] Loading Test Data (All Outage Dates)
--------------------------------------------------------------------------------
Test period: 2025-10
  Scanning for outage dates in 2025-10...
  ✓ 2025-10-01: 12,646 constraints
  ✓ 2025-10-04: 12,691 constraints
  ✓ 2025-10-07: 12,642 constraints
  ✓ 2025-10-10: 12,743 constraints
  ✓ 2025-10-13: 12,621 constraints
  ✓ 2025-10-16: 12,729 constraints
  ✓ 2025-10-19: 12,767 constraints
  ✓ 2025-10-22: 12,751 constraints
  ✓ 2025-10-25: 12,787 constraints
  ✓ 2025-10-28: 12,765 constraints
  ✓ 2025-10-31: 12,795 constraints

Combining 11 test datasets...

✓ Test Data Loaded Successfully
  Total samples: 139,937
  Outage dates loaded: 11
  Date range: 2025-10-01 to 2025-10-31
  Unique constraints: 13,048

  Class Distribution:
    Binding: 5,345 (3.82%)
    Non-binding: 134,592 (96.18%)

  Samples per outage date:
    2025-10-01: 12,646 samples (533 binding)
    2025-10-04: 12,691 samples (269 binding)
    2025-10-07: 12,642 samples (575 bin

In [6]:
# ============================================================================
# STEP 5: Prepare Training and Test Datasets (BY BRANCH_NAME MODELS)
# ============================================================================

print("\n[STEP 5] Preparing Features and Targets (Train separate models per branch_name)")
print("-" * 80)

if train_data_combined is not None and test_data is not None:
    
    # ========================================================================
    # Training data: Keep at constraint_id level but group by branch_name
    # ========================================================================
    print("Organizing training data by branch_name...")
    
    # Group training data by branch_name to prepare for model training
    unique_branch_names_train = train_data_combined['branch_name'].unique()
    n_branches_train = len(unique_branch_names_train)
    
    print(f"  Training samples: {len(train_data_combined):,}")
    print(f"  Unique branch_names in training: {n_branches_train:,}")
    
    # Create a dict to store training data by branch_name
    train_data_by_branch = {branch_name: branch_data for branch_name, branch_data in train_data_combined.groupby('branch_name')}
    
    print(f"  Organized into {len(train_data_by_branch):,} branch groups")
    
    # Show sample statistics
    sample_sizes = [len(v) for v in train_data_by_branch.values()]
    print(f"  Training samples per branch: min={min(sample_sizes)}, max={max(sample_sizes)}, avg={np.mean(sample_sizes):.1f}")
    
    # ========================================================================
    # Test data: Keep at constraint_id level (now includes multiple outage dates)
    # ========================================================================
    print("\nPreparing test data...")
    
    X_test_prod = test_data[['score']].copy()
    y_test_prod = test_data['label'].copy()
    y_test_binary_prod = (y_test_prod > 0).astype(int)
    
    # Extract metadata for each test sample
    test_branch_names = test_data['branch_name'].values
    test_outage_dates = test_data['outage_date'].values
    test_auction_months = test_data['auction_month'].values
    test_market_months = test_data['market_month'].values
    
    # Get unique branch names in test set
    unique_branch_names_test = test_data['branch_name'].unique()
    n_branches_test = len(unique_branch_names_test)
    
    # Get unique outage dates in test set
    unique_outage_dates_test = test_data['outage_date'].unique()
    n_outage_dates_test = len(unique_outage_dates_test)
    
    print(f"  Test samples: {len(test_data):,}")
    print(f"  Unique outage dates: {n_outage_dates_test}")
    print(f"  Unique branch_names in test: {n_branches_test:,}")
    
    # Check overlap between train and test branch names
    common_branches = set(unique_branch_names_train) & set(unique_branch_names_test)
    test_only_branches = set(unique_branch_names_test) - set(unique_branch_names_train)
    
    print(f"\n  Branch overlap:")
    print(f"    Common branches (train & test): {len(common_branches):,}")
    print(f"    Test-only branches (no training data): {len(test_only_branches):,}")
    
    if len(test_only_branches) > 0:
        print(f"    ⚠️  Warning: {len(test_only_branches)} test branches have no training data")
        print(f"       These will use a default fallback model")
    
    print(f"\n✓ Test Data:")
    print(f"  Features (X_test): {X_test_prod.shape}")
    print(f"  Targets (y_test): {y_test_prod.shape}")
    print(f"  Binding rate: {y_test_binary_prod.sum()/len(y_test_binary_prod)*100:.2f}%")
    print(f"  Covering {n_outage_dates_test} outage dates from {test_data['outage_date'].min().strftime('%Y-%m-%d')} to {test_data['outage_date'].max().strftime('%Y-%m-%d')}")
    
else:
    print("⚠️  ERROR: Cannot prepare data - training or test data missing")


[STEP 5] Preparing Features and Targets (Train separate models per branch_name)
--------------------------------------------------------------------------------
Organizing training data by branch_name...
  Training samples: 1,593,377
  Unique branch_names in training: 4,469
  Organized into 4,469 branch groups
  Training samples per branch: min=2, max=4455, avg=356.5

Preparing test data...
  Test samples: 139,937
  Unique outage dates: 11
  Unique branch_names in test: 4,206

  Branch overlap:
    Common branches (train & test): 4,161
    Test-only branches (no training data): 45
    ⚠️  Warning: 45 test branches have no training data
       These will use a default fallback model

✓ Test Data:
  Features (X_test): (139937, 1)
  Targets (y_test): (139937,)
  Binding rate: 3.82%
  Covering 11 outage dates from 2025-10-01 to 2025-10-31


In [7]:
# ============================================================================
# STEP 6: Train Separate Models for Each Branch (Stage 1: Classification)
#         WITH DYNAMIC THRESHOLD OPTIMIZATION
# ============================================================================

print("\n[STEP 6] Training Branch-Specific Classification Models + Threshold Optimization")
print("-" * 80)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

def find_optimal_threshold(y_true, y_proba, thresholds=None):
    """
    Find optimal classification threshold that maximizes F1-score.
    
    Parameters:
    -----------
    y_true : array-like
        True binary labels
    y_proba : array-like
        Predicted probabilities for positive class
    thresholds : array-like, optional
        Threshold values to search (default: np.linspace(0.01, 0.99, 99))
    
    Returns:
    --------
    optimal_threshold : float
        Threshold that maximizes F1-score
    max_f1 : float
        Maximum F1-score achieved
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)
    
    f1_scores = []
    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        f1_scores.append(f1)
    
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]
    max_f1 = f1_scores[optimal_idx]
    
    return optimal_threshold, max_f1


if train_data_combined is not None:
    
    # Dictionary to store models and thresholds by branch_name
    clf_models = {}
    optimal_thresholds = {}  # NEW: Store optimal thresholds
    
    # Also train a default/fallback model on all data for branches not seen in training
    print("Training default fallback classifier on all training data...")
    
    X_train_all = train_data_combined[['score']].copy()
    y_train_all = train_data_combined['label'].copy()
    y_train_binary_all = (y_train_all > 0).astype(int)
    
    # Calculate class weights for all data
    n_samples_all = len(y_train_binary_all)
    n_binding_all = y_train_binary_all.sum()
    n_non_binding_all = n_samples_all - n_binding_all
    
    weight_binding_all = n_samples_all / (2 * n_binding_all) if n_binding_all > 0 else 1.0
    weight_non_binding_all = n_samples_all / (2 * n_non_binding_all)
    class_weights_all = {0: weight_non_binding_all, 1: weight_binding_all}
    
    clf_default = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=10,
        class_weight=class_weights_all,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    clf_default.fit(X_train_all, y_train_binary_all)
    print(f"  ✓ Default classifier trained on {len(X_train_all):,} samples")
    
    # NEW: Optimize threshold for default classifier
    print(f"  Optimizing threshold for default classifier...")
    y_proba_train_all = clf_default.predict_proba(X_train_all)[:, 1]
    optimal_threshold_default, max_f1_default = find_optimal_threshold(
        y_train_binary_all, y_proba_train_all
    )
    print(f"  ✓ Optimal threshold (default): {optimal_threshold_default:.3f} (F1={max_f1_default:.3f})")
    
    # Now train branch-specific models
    print(f"\nTraining branch-specific classifiers + optimizing thresholds...")
    print(f"  Total branches to train: {len(train_data_by_branch):,}")
    
    trained_count = 0
    skipped_count = 0
    min_samples_required = 10  # Minimum samples needed to train a model
    
    total_test_branches = set(test_data['branch_name'].unique())
    for branch_name, branch_data in train_data_by_branch.items():
        if branch_name not in total_test_branches:
            skipped_count += 1
            continue
        
        if len(branch_data) < min_samples_required:
            skipped_count += 1
            continue
        
        # Extract features and targets
        X_branch = branch_data[['score']].copy()
        y_branch = branch_data['label'].copy()
        y_branch_binary = (y_branch > 0).astype(int)

        if y_branch_binary.sum() == 0:
            skipped_count += 1
            continue
        
        # Calculate class weights for this branch
        n_samples_branch = len(y_branch_binary)
        n_binding_branch = y_branch_binary.sum()
        n_non_binding_branch = n_samples_branch - n_binding_branch
        
        if n_binding_branch > 0 and n_non_binding_branch > 0:
            weight_binding_branch = n_samples_branch / (2 * n_binding_branch)
            weight_non_binding_branch = n_samples_branch / (2 * n_non_binding_branch)
            class_weights_branch = {0: weight_non_binding_branch, 1: weight_binding_branch}
        else:
            # If only one class, use balanced weights
            class_weights_branch = 'balanced'
        
        # Train classifier for this branch
        clf_branch = RandomForestClassifier(
            n_estimators=50,  # Fewer trees for individual branches
            max_depth=8,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight=class_weights_branch,
            random_state=42,
            n_jobs=1,
            verbose=0
        )
        
        try:
            clf_branch.fit(X_branch, y_branch_binary)
            clf_models[branch_name] = clf_branch
            
            # NEW: Optimize threshold for this branch
            y_proba_train_branch = clf_branch.predict_proba(X_branch)[:, 1]
            optimal_threshold_branch, max_f1_branch = find_optimal_threshold(
                y_branch_binary, y_proba_train_branch
            )
            optimal_thresholds[branch_name] = optimal_threshold_branch
            
            trained_count += 1
        except:
            skipped_count += 1
            continue
        
        # Print progress every 1000 models
        if trained_count % 1000 == 0:
            print(f"    Progress: {trained_count:,} models trained...")
    
    print(f"\n✓ Branch-Specific Model Training Complete")
    print(f"  Models trained: {trained_count:,}")
    print(f"  Branches skipped (insufficient data): {skipped_count:,}")
    print(f"  Default fallback model: Available for unseen branches")
    
    # NEW: Show threshold statistics
    print(f"\n[THRESHOLD OPTIMIZATION STATISTICS]")
    print(f"  Default threshold: {optimal_threshold_default:.3f}")
    
    if len(optimal_thresholds) > 0:
        threshold_values = list(optimal_thresholds.values())
        print(f"  Branch-specific thresholds:")
        print(f"    Count: {len(threshold_values):,}")
        print(f"    Mean: {np.mean(threshold_values):.3f}")
        print(f"    Median: {np.median(threshold_values):.3f}")
        print(f"    Std: {np.std(threshold_values):.3f}")
        print(f"    Min: {np.min(threshold_values):.3f}")
        print(f"    Max: {np.max(threshold_values):.3f}")
        
        # Show distribution
        below_05 = sum(1 for t in threshold_values if t < 0.5)
        above_05 = sum(1 for t in threshold_values if t >= 0.5)
        print(f"  Distribution:")
        print(f"    Thresholds < 0.5: {below_05:,} ({below_05/len(threshold_values)*100:.1f}%)")
        print(f"    Thresholds ≥ 0.5: {above_05:,} ({above_05/len(threshold_values)*100:.1f}%)")
    
else:
    print("⚠️  ERROR: No training data available")


[STEP 6] Training Branch-Specific Classification Models + Threshold Optimization
--------------------------------------------------------------------------------
Training default fallback classifier on all training data...
  ✓ Default classifier trained on 1,593,377 samples
  Optimizing threshold for default classifier...
  ✓ Optimal threshold (default): 0.760 (F1=0.187)

Training branch-specific classifiers + optimizing thresholds...
  Total branches to train: 4,469

✓ Branch-Specific Model Training Complete
  Models trained: 831
  Branches skipped (insufficient data): 3,638
  Default fallback model: Available for unseen branches

[THRESHOLD OPTIMIZATION STATISTICS]
  Default threshold: 0.760
  Branch-specific thresholds:
    Count: 831
    Mean: 0.531
    Median: 0.540
    Std: 0.120
    Min: 0.010
    Max: 0.930
  Distribution:
    Thresholds < 0.5: 229 (27.6%)
    Thresholds ≥ 0.5: 602 (72.4%)


In [8]:
# ============================================================================
# STEP 7: Train Separate Regression Models for Each Branch (Stage 2)
# ============================================================================

print("\n[STEP 7] Training Branch-Specific Regression Models")
print("-" * 80)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

if train_data_combined is not None:
    
    # Dictionary to store regression models by branch_name
    reg_models = {}
    
    # Train default fallback regressor on all binding data
    print("Training default fallback regressor on all binding training data...")
    
    binding_mask_all = y_train_all > 0
    X_train_binding_all = X_train_all[binding_mask_all]
    y_train_binding_all = y_train_all[binding_mask_all]
    
    if len(X_train_binding_all) > 10:
        reg_default = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1,
            verbose=0
        )
        reg_default.fit(X_train_binding_all, y_train_binding_all)
        print(f"  ✓ Default regressor trained on {len(X_train_binding_all):,} binding samples")
    else:
        reg_default = None
        print(f"  ⚠️  Insufficient binding samples for default regressor")
    
    # Now train branch-specific regression models
    print(f"\nTraining branch-specific regressors...")
    print(f"  Total branches to train: {len(train_data_by_branch):,}")
    
    trained_reg_count = 0
    skipped_reg_count = 0
    min_binding_samples_required = 1  # Minimum binding samples for regression
    
    for branch_name, branch_data in train_data_by_branch.items():
        if branch_name not in total_test_branches:
            skipped_reg_count += 1
            continue
        
        # Extract binding samples for this branch
        X_branch = branch_data[['score']].copy()
        y_branch = branch_data['label'].copy()
        
        binding_mask_branch = y_branch > 0
        X_branch_binding = X_branch[binding_mask_branch]
        y_branch_binding = y_branch[binding_mask_branch]
        
        # Skip if not enough binding samples
        if len(X_branch_binding) < min_binding_samples_required:
            skipped_reg_count += 1
            continue
        
        # Train regressor for this branch
        reg_branch = RandomForestRegressor(
            n_estimators=50,
            max_depth=8,
            min_samples_split=3,
            min_samples_leaf=1,
            random_state=42,
            n_jobs=1,
            verbose=0
        )
        
        try:
            reg_branch.fit(X_branch_binding, y_branch_binding)
            reg_models[branch_name] = reg_branch
            trained_reg_count += 1
        except:
            skipped_reg_count += 1
            continue
        
        # Print progress every 500 models
        if trained_reg_count % 500 == 0:
            print(f"    Progress: {trained_reg_count:,} regression models trained...")
    
    print(f"\n✓ Branch-Specific Regression Training Complete")
    print(f"  Regression models trained: {trained_reg_count:,}")
    print(f"  Branches skipped (insufficient binding data): {skipped_reg_count:,}")
    print(f"  Default fallback regressor: {'Available' if reg_default is not None else 'Not available'}")
    
else:
    print("⚠️  ERROR: No training data available")


[STEP 7] Training Branch-Specific Regression Models
--------------------------------------------------------------------------------
Training default fallback regressor on all binding training data...
  ✓ Default regressor trained on 46,563 binding samples

Training branch-specific regressors...
  Total branches to train: 4,469
    Progress: 500 regression models trained...

✓ Branch-Specific Regression Training Complete
  Regression models trained: 832
  Branches skipped (insufficient binding data): 3,637
  Default fallback regressor: Available


In [9]:
# ============================================================================
# STEP 8: Make Predictions Using Branch-Specific Models (BATCH PROCESSING)
#         WITH DYNAMIC THRESHOLDS
# ============================================================================

print("\n[STEP 8] Making Predictions with Branch-Specific Models (Batch Processing + Dynamic Thresholds)")
print("=" * 80)

# Initialize prediction arrays
y_pred_binary_prod = np.zeros(len(X_test_prod), dtype=int)
y_pred_proba_prod = np.zeros(len(X_test_prod))
y_pred_shadow_price_prod = np.zeros(len(X_test_prod))

# Track which model was used for each prediction
model_used = [''] * len(X_test_prod)

# Get unique branch names in test set
unique_branches_in_test = test_data['branch_name'].unique()

print(f"\nProcessing {len(unique_branches_in_test):,} unique branches (batch predictions per branch)...")
print(f"Total test samples: {len(X_test_prod):,}")

used_branch_clf_count = 0
used_default_clf_count = 0
used_branch_reg_count = 0
used_default_reg_count = 0
no_reg_model_count = 0

branches_processed = 0

for branch_name in unique_branches_in_test:
    
    # Get all test samples for this branch_name
    branch_mask = test_data['branch_name'] == branch_name
    branch_indices = np.where(branch_mask)[0]
    
    # Get features for all samples in this branch
    X_branch = X_test_prod.iloc[branch_indices]
    n_samples_in_branch = len(X_branch)
    
    # ========================================================================
    # Stage 1: Classification (Binding Detection) - BATCH WITH DYNAMIC THRESHOLD
    # ========================================================================
    if branch_name in clf_models:
        # Use branch-specific classifier
        clf_to_use = clf_models[branch_name]
        used_branch_clf_count += n_samples_in_branch
        model_name = f"Branch: {branch_name}"
        
        # NEW: Get optimal threshold for this branch
        threshold_to_use = optimal_thresholds.get(branch_name, optimal_threshold_default)
    else:
        # Use default classifier
        clf_to_use = clf_default
        used_default_clf_count += n_samples_in_branch
        model_name = "Default"
        
        # NEW: Use default optimal threshold
        threshold_to_use = optimal_threshold_default
    
    # NEW: Get probabilities and apply dynamic threshold
    y_pred_proba_branch = clf_to_use.predict_proba(X_branch)[:, 1]
    y_pred_binary_branch = (y_pred_proba_branch >= threshold_to_use).astype(int)
    
    # Store predictions
    y_pred_binary_prod[branch_indices] = y_pred_binary_branch
    y_pred_proba_prod[branch_indices] = np.where(
        y_pred_binary_branch <= threshold_to_use,
        (0.5 / threshold_to_use) * y_pred_proba_branch,
        0.5 + (0.5 / (1 - threshold_to_use)) * (y_pred_proba_branch - threshold_to_use)
    )
    
    # ========================================================================
    # Stage 2: Regression (Shadow Price Prediction) - BATCH for binding only
    # ========================================================================
    # Find which samples in this branch were predicted as binding
    binding_mask_in_branch = y_pred_binary_branch == 1
    
    if binding_mask_in_branch.sum() > 0:
        # Get indices of binding samples within this branch
        binding_indices_in_branch = branch_indices[binding_mask_in_branch]
        X_binding_in_branch = X_branch.iloc[binding_mask_in_branch]
        
        if branch_name in reg_models:
            # Use branch-specific regressor
            reg_to_use = reg_models[branch_name]
            used_branch_reg_count += len(binding_indices_in_branch)
        elif reg_default is not None:
            # Use default regressor
            reg_to_use = reg_default
            used_default_reg_count += len(binding_indices_in_branch)
        else:
            # No regressor available
            reg_to_use = None
            no_reg_model_count += len(binding_indices_in_branch)
        
        # Batch prediction for regression
        if reg_to_use is not None:
            y_pred_shadow_price_branch = reg_to_use.predict(X_binding_in_branch)
            y_pred_shadow_price_prod[binding_indices_in_branch] = y_pred_shadow_price_branch
    
    # Store model name for all samples in this branch
    for idx in branch_indices:
        model_used[idx] = model_name
    
    branches_processed += 1
    
    # Progress indicator
    if branches_processed % 1000 == 0:
        print(f"  Progress: {branches_processed:,} / {len(unique_branches_in_test):,} branches processed...")

print(f"\n✓ Predictions Complete")
print(f"\n[Model Usage Statistics]")
print(f"  Classification:")
print(f"    Branch-specific models: {used_branch_clf_count:,} samples ({used_branch_clf_count/len(X_test_prod)*100:.2f}%)")
print(f"    Default fallback model: {used_default_clf_count:,} samples ({used_default_clf_count/len(X_test_prod)*100:.2f}%)")
print(f"  Regression (for predicted binding):")
print(f"    Branch-specific models: {used_branch_reg_count:,} samples")
print(f"    Default fallback model: {used_default_reg_count:,} samples")
print(f"    No model available: {no_reg_model_count:,} samples")
print(f"\n  Dynamic Thresholds Applied:")
print(f"    Default threshold: {optimal_threshold_default:.3f}")
print(f"    Branch-specific thresholds used: {sum(1 for b in unique_branches_in_test if b in optimal_thresholds):,}")
print(f"    Branches using default threshold: {sum(1 for b in unique_branches_in_test if b not in optimal_thresholds):,}")


[STEP 8] Making Predictions with Branch-Specific Models (Batch Processing + Dynamic Thresholds)

Processing 4,206 unique branches (batch predictions per branch)...
Total test samples: 139,937
  Progress: 1,000 / 4,206 branches processed...
  Progress: 2,000 / 4,206 branches processed...
  Progress: 3,000 / 4,206 branches processed...
  Progress: 4,000 / 4,206 branches processed...

✓ Predictions Complete

[Model Usage Statistics]
  Classification:
    Branch-specific models: 48,305 samples (34.52%)
    Default fallback model: 91,632 samples (65.48%)
  Regression (for predicted binding):
    Branch-specific models: 4,716 samples
    Default fallback model: 3,393 samples
    No model available: 0 samples

  Dynamic Thresholds Applied:
    Default threshold: 0.760
    Branch-specific thresholds used: 831
    Branches using default threshold: 3,375


In [10]:
# ============================================================================
# AGGREGATE PREDICTIONS ACROSS ALL OUTAGE DATES
# ============================================================================

print("\n" + "=" * 80)
print("[AGGREGATING PREDICTIONS ACROSS ALL OUTAGE DATES]")
print("=" * 80)

# Create detailed results DataFrame (per outage date, per constraint)
results_per_outage = test_data[['score', 'label', 'branch_name', 'outage_date', 'auction_month', 'market_month']].copy()
results_per_outage['predicted_shadow_price'] = y_pred_shadow_price_prod
results_per_outage['binding_probability'] = y_pred_proba_prod
results_per_outage['predicted_binding'] = y_pred_binary_prod
results_per_outage['actual_binding'] = (test_data['label'] > 0).astype(int).values
results_per_outage = results_per_outage.rename(columns={'label': 'actual_shadow_price'})

print(f"\nBefore aggregation:")
print(f"  Total samples (constraint × outage_date): {len(results_per_outage):,}")
print(f"  Unique constraints: {results_per_outage.index.get_level_values(0).nunique():,}")
print(f"  Unique outage dates: {results_per_outage['outage_date'].nunique()}")

# Aggregate by constraint_id (sum across all outage dates)
final_results_prod = results_per_outage.groupby(level=[0, 1]).agg({
    'branch_name': 'first',  # Same for all outage dates
    'score': 'mean',  # Average score across outage dates
    'actual_shadow_price': 'sum',  # SUM of actual shadow prices
    'predicted_shadow_price': 'sum',  # SUM of predicted shadow prices
    'binding_probability': 'max',  # Maximum binding probability across dates
    'predicted_binding': 'max',  # Binding if predicted binding in ANY outage date
    'actual_binding': 'max',  # Binding if actually binding in ANY outage date
    'auction_month': 'first',
    'market_month': 'first'
})

# Calculate errors on aggregated data
final_results_prod['error'] = final_results_prod['predicted_shadow_price'] - final_results_prod['actual_shadow_price']
final_results_prod['abs_error'] = np.abs(final_results_prod['error'])

# Add model used (get from first outage date)
model_used_agg = results_per_outage.groupby(level=[0, 1])['outage_date'].first().map(
    lambda x: results_per_outage[results_per_outage['outage_date'] == x].iloc[0] if len(results_per_outage[results_per_outage['outage_date'] == x]) > 0 else 'Unknown'
)
# Simplified: just use branch_name to determine model type
final_results_prod['model_used'] = final_results_prod['branch_name'].apply(
    lambda b: f"Branch: {b}" if b in clf_models else "Default"
)

print(f"\nAfter aggregation:")
print(f"  Total unique constraints: {len(final_results_prod):,}")
print(f"  Aggregation: SUM of shadow prices across {results_per_outage['outage_date'].nunique()} outage dates")
print(f"  Binding definition: Constraint binds if it binds in ANY outage date")

# Calculate overall metrics on AGGREGATED data
y_test_agg = final_results_prod['actual_shadow_price'].values
y_pred_agg = final_results_prod['predicted_shadow_price'].values
y_test_binary_agg = final_results_prod['actual_binding'].values
y_pred_binary_agg = final_results_prod['predicted_binding'].values

mae_overall_prod = mean_absolute_error(y_test_agg, y_pred_agg)
rmse_overall_prod = np.sqrt(mean_squared_error(y_test_agg, y_pred_agg))

print(f"\n[Overall Two-Stage Model Performance on AGGREGATED Data]")
print(f"  Test Period: {results_per_outage['outage_date'].min().strftime('%Y-%m-%d')} to {results_per_outage['outage_date'].max().strftime('%Y-%m-%d')}")
print(f"  Total unique constraints: {len(final_results_prod):,}")
print(f"  MAE (aggregated):  ${mae_overall_prod:.2f}")
print(f"  RMSE (aggregated): ${rmse_overall_prod:.2f}")

# Classification metrics on AGGREGATED data
f1_prod = f1_score(y_test_binary_agg, y_pred_binary_agg)
precision_prod = precision_score(y_test_binary_agg, y_pred_binary_agg, zero_division=0)
recall_prod = recall_score(y_test_binary_agg, y_pred_binary_agg, zero_division=0)

print(f"\n[Classification Performance on AGGREGATED Data]")
print(f"  Precision: {precision_prod:.3f}")
print(f"  Recall:    {recall_prod:.3f}")
print(f"  F1-Score:  {f1_prod:.3f}")

print("\n" + "=" * 80)
print("[FINAL PREDICTION RESULTS - Top 15 by Predicted Shadow Price (AGGREGATED)]")
print("=" * 80)

top_15_predictions = final_results_prod.nlargest(15, 'predicted_shadow_price')
print(top_15_predictions[['branch_name', 'score', 'binding_probability', 'predicted_shadow_price', 
                          'actual_shadow_price', 'error']].to_string())

print("\n" + "=" * 80)
print("[PREDICTION ACCURACY SUMMARY - AGGREGATED DATA]")
print("=" * 80)

# Binding classification accuracy on aggregated data
correct_binding = (final_results_prod['actual_binding'] == final_results_prod['predicted_binding']).sum()
binding_accuracy = correct_binding / len(final_results_prod) * 100

print(f"Binding Classification (Aggregated):")
print(f"  Correctly classified: {correct_binding:,} / {len(final_results_prod):,} ({binding_accuracy:.2f}%)")

# True positives, false positives, etc.
tp = ((final_results_prod['actual_binding'] == 1) & (final_results_prod['predicted_binding'] == 1)).sum()
fp = ((final_results_prod['actual_binding'] == 0) & (final_results_prod['predicted_binding'] == 1)).sum()
tn = ((final_results_prod['actual_binding'] == 0) & (final_results_prod['predicted_binding'] == 0)).sum()
fn = ((final_results_prod['actual_binding'] == 1) & (final_results_prod['predicted_binding'] == 0)).sum()

print(f"\n  True Positives (correctly identified binding): {tp:,}")
print(f"  False Positives (incorrectly predicted as binding): {fp:,}")
print(f"  True Negatives (correctly identified non-binding): {tn:,}")
print(f"  False Negatives (missed binding constraints): {fn:,}")

# Error distribution on aggregated data
print(f"\nShadow Price Prediction Errors (Aggregated):")
print(f"  Mean Absolute Error: ${final_results_prod['abs_error'].mean():.2f}")
print(f"  Median Absolute Error: ${final_results_prod['abs_error'].median():.2f}")
print(f"  Max Absolute Error: ${final_results_prod['abs_error'].max():.2f}")

print("\n" + "=" * 80)
print("✅ PREDICTION WORKFLOW COMPLETE!")
print("=" * 80)
print(f"\n💡 Performance Improvement: Batch processing by branch_name")
print(f"   - Processed {len(unique_branches_in_test):,} branches instead of {len(X_test_prod):,} individual samples")
print(f"   - Significant speedup from vectorized predictions!")
print(f"\n📊 Results aggregated across {results_per_outage['outage_date'].nunique()} outage dates")
print(f"   - Analysis based on summed shadow prices per constraint")


[AGGREGATING PREDICTIONS ACROSS ALL OUTAGE DATES]

Before aggregation:
  Total samples (constraint × outage_date): 139,937
  Unique constraints: 13,048
  Unique outage dates: 11

After aggregation:
  Total unique constraints: 15,543
  Aggregation: SUM of shadow prices across 11 outage dates
  Binding definition: Constraint binds if it binds in ANY outage date

[Overall Two-Stage Model Performance on AGGREGATED Data]
  Test Period: 2025-10-01 to 2025-10-31
  Total unique constraints: 15,543
  MAE (aggregated):  $542.51
  RMSE (aggregated): $2526.77

[Classification Performance on AGGREGATED Data]
  Precision: 0.325
  Recall:    0.517
  F1-Score:  0.399

[FINAL PREDICTION RESULTS - Top 15 by Predicted Shadow Price (AGGREGATED)]
                                                        branch_name     score  binding_probability  predicted_shadow_price  actual_shadow_price         error
constraint_id                     flow_direction                                                         

In [11]:
signal = pd.read_parquet('/opt/data/xyz-dataset/signal_data/miso/constraints/TEST.TEST.Signal.MISO.SPICE_F0P_V6.2B.R1/2025-10/f0/onpeak/')
final_results_prod['signal_tier'] = final_results_prod['branch_name'].map(signal.groupby('equipment')['tier'].min())

xx = final_results_prod.sort_values('predicted_shadow_price', ascending=False).drop_duplicates(subset=['branch_name'])
xx.sort_values('actual_shadow_price', ascending=False).head(20)

,,branch_name,score,actual_shadow_price,predicted_shadow_price,binding_probability,predicted_binding,actual_binding,auction_month,market_month,error,abs_error,model_used,signal_tier
constraint_id,flow_direction,,,,,,,,,,,,,
329180,1,FORMAFORMN11_1 1,4.939090e-03,59113.56,14102.577930,0.662083,1,1,2025-10-01,2025-10-01,-45010.982070,45010.982070,Branch: FORMAFORMN11_1 1,0.0
373761,-1,GOOSEMNPIP11_1 1,1.779110e-12,49015.21,0.000000,0.288105,0,1,2025-10-01,2025-10-01,-49015.210000,49015.210000,Default,NaN
121700,-1,ULRICMAHNO11_1 1,2.463417e-02,26006.01,73.120000,0.534974,1,1,2025-10-01,2025-10-01,-25932.890000,25932.890000,Branch: ULRICMAHNO11_1 1,0.0
226064,1,RILLA__RIVTON3 A,1.525925e-02,25956.94,4603.901557,0.586639,1,1,2025-10-01,2025-10-01,-21353.038443,21353.038443,Branch: RILLA__RIVTON3 A,0.0
430652,1,ROSEHURON23_1 1,1.961501e-02,25655.66,7098.944800,0.749620,1,1,2025-10-01,2025-10-01,-18556.715200,18556.715200,Branch: ROSEHURON23_1 1,NaN
28297,1,PENN_LEEDS11_1 1,5.622619e-02,19450.78,3082.050326,0.790008,1,1,2025-10-01,2025-10-01,-16368.729674,16368.729674,Branch: PENN_LEEDS11_1 1,NaN
521,1,MORRIGRANT11_1 1,2.449514e-02,19258.65,4631.600750,0.566994,1,1,2025-10-01,2025-10-01,-14627.049250,14627.049250,Branch: MORRIGRANT11_1 1,0.0
415938,1,MAPLE08CHR13_1 1,2.297221e-01,16221.99,13864.335199,0.695112,1,1,2025-10-01,2025-10-01,-2357.654801,2357.654801,Branch: MAPLE08CHR13_1 1,0.0
332658,-1,BIGSTBROWN23_1 1,1.007136e-01,15381.76,8170.706730,0.935757,1,1,2025-10-01,2025-10-01,-7211.053270,7211.053270,Branch: BIGSTBROWN23_1 1,0.0


In [12]:
xx[xx['branch_name'].str.contains('AUST_TAYS_1545')]

,,branch_name,score,actual_shadow_price,predicted_shadow_price,binding_probability,predicted_binding,actual_binding,auction_month,market_month,error,abs_error,model_used,signal_tier
constraint_id,flow_direction,,,,,,,,,,,,,
316316,1,AUST_TAYS_1545 A,0.235311,2950.54,9004.441832,0.77529,1,1,2025-10-01,2025-10-01,6053.901832,6053.901832,Default,0.0


In [ ]:
final_results_prod

---

# 📋 Quick Reference: Production Workflow Summary (Branch-Specific Models)

## Complete Workflow Steps (Run in Order)

| Step | Cell | Action | Output |
|------|------|--------|--------|
| **1** | Define date ranges | Set training (2024-11 to 2025-09) and test month (2025-11, ALL outage dates) | Date parameters |
| **2** | Helper function | Define `load_training_data_for_outage()` | Data loading function |
| **3** | Load training data | Load 11 months of historical data | `train_data_combined` |
| **4** | **Load ALL test data** | **Load ALL outage dates in test month (2025-11)** | **`test_data` (multiple outage dates)** |
| **5** | **Organize by branch** | **Group training data by branch_name** | **`train_data_by_branch` dict** |
| **6** | **Train classifiers** | **Train separate classifier for each branch** | **`clf_models` dict + `clf_default`** |
| **7** | **Train regressors** | **Train separate regressor for each branch** | **`reg_models` dict + `reg_default`** |
| **8** | Make predictions | Use branch-specific models for each constraint across all outage dates | `final_results_prod` DataFrame |
| **9** | Visualize | Create 4 plots (aggregated across all outage dates) | Visualizations |
| **10** | Export results | Save 3 CSV files (constraint-level, branch summary, outage date summary) | 3 CSV files |

## 🔑 Key Strategy: Branch-Specific Models + Multiple Outage Dates

### Training Strategy

**For each unique branch_name:**
1. Extract all historical constraint_id records with that branch_name
2. Train a dedicated classifier on those records
3. Train a dedicated regressor on binding samples from those records
4. Store models in dictionaries: `clf_models[branch_name]`, `reg_models[branch_name]`

**Example:**
```
Branch: "LINE_A"
  Training data (from 2024-11 to 2025-09):
    - 116 outage periods × N constraints with branch_name="LINE_A"
    - Total: ~150 samples for this branch
    
  Models trained:
    - clf_models["LINE_A"]: Classifier trained on all "LINE_A" records
    - reg_models["LINE_A"]: Regressor trained on binding "LINE_A" records
```

### Prediction Strategy (Multiple Outage Dates)

**Test Period: 2025-11 (ALL outage dates)**
- Automatically loads all available outage dates in the month (typically 10-11 dates)
- Makes predictions for EVERY constraint at EVERY outage date
- Same models used across all outage dates

**For each constraint_id at each outage_date:**
1. Look up its branch_name
2. Use `clf_models[branch_name]` if available, else use `clf_default`
3. If predicted as binding, use `reg_models[branch_name]` if available, else use `reg_default`
4. Return individual prediction for this (constraint_id, outage_date) pair

**Example:**
```
Test Month: 2025-11
  Outage dates found: 2025-11-01, 2025-11-04, 2025-11-07, ..., 2025-11-28 (10 dates)
  
For constraint_id="ABC123" with branch_name="LINE_A":
  → 2025-11-01: Use clf_models["LINE_A"] → predict binding & shadow price
  → 2025-11-04: Use clf_models["LINE_A"] → predict binding & shadow price
  → 2025-11-07: Use clf_models["LINE_A"] → predict binding & shadow price
  ... (10 predictions total for this constraint)
```

### Why This Approach?

✅ **Reusable Models**: Train once, predict for all outage dates in the month  
✅ **Consistent Predictions**: Same model logic across all dates  
✅ **Comprehensive Coverage**: No manual iteration through outage dates  
✅ **Production Ready**: Natural workflow for monthly prediction runs  

## Key Outputs

### Trained Models (Dictionaries)
- `clf_models`: `{branch_name: RandomForestClassifier}` - One classifier per branch
- `reg_models`: `{branch_name: RandomForestRegressor}` - One regressor per branch
- `clf_default`: Fallback classifier for unseen branches
- `reg_default`: Fallback regressor for unseen branches

### Results DataFrames
- `final_results_prod`: Complete predictions **indexed by (constraint_id, flow_direction)**
  - Columns: 
    - `outage_date`: Which outage date this prediction is for
    - `auction_month`, `market_month`: Market context
    - `branch_name`: Which branch this constraint belongs to
    - `score`: Feature value for this constraint
    - `actual_shadow_price`: Ground truth shadow price
    - `predicted_shadow_price`: Model prediction
    - `binding_probability`: Probability of binding
    - `predicted_binding`, `actual_binding`: Binary predictions
    - `error`, `abs_error`: Prediction errors
    - `model_used`: Which model was used for this prediction

### Exported Files (3 CSV Files)
1. **`shadow_price_predictions_branch_models_YYYYMM.csv`**: 
   - Individual constraint predictions for EVERY outage date
   - Format: (constraint_id, flow_direction, outage_date) → predictions
   
2. **`branch_model_summary_YYYYMM.csv`**: 
   - Aggregated statistics by branch_name (across ALL outage dates)
   - Shows total shadow prices, average errors, binding counts per branch
   
3. **`outage_date_summary_YYYYMM.csv`**: 
   - Aggregated statistics by outage_date
   - Shows performance metrics for each individual outage date

## Model Architecture

```
Training Phase (Once):
  For each branch_name in training data:
    ├─ Extract all constraint records with this branch_name
    ├─ Train classifier: clf_models[branch_name]
    └─ Train regressor: reg_models[branch_name] (on binding samples)
  
  Also train default models:
    ├─ clf_default (on all training data)
    └─ reg_default (on all binding data)

Prediction Phase (All Outage Dates):
  Load all outage dates in test month → test_data
  
  For each constraint in test_data (across ALL outage dates):
    ├─ Get branch_name
    ├─ Classification:
    │   ├─ If branch_name in clf_models → use clf_models[branch_name]
    │   └─ Else → use clf_default
    ├─ Regression (if predicted binding):
    │   ├─ If branch_name in reg_models → use reg_models[branch_name]
    │   └─ Else → use reg_default
    └─ Return prediction for this (constraint_id, outage_date)
```

## Performance Expectations

**Typical Results for a Test Month:**
- Outage dates loaded: 10-11 (every 3 days in the month)
- Test samples: 120,000-140,000 (12,000-13,000 constraints × 10-11 dates)
- Classifiers trained per branch: ~10,000-12,000
- Regressors trained per branch: ~500-2,000
- Predictions using branch models: >95%
- Predictions using default models: <5%

## How to Modify for Different Periods

**Change test period in Cell 1 (STEP 1):**

```python
# Example 1: Predict for December 2025
test_auction_month = pd.Timestamp('2025-12')
test_market_month = pd.Timestamp('2025-12')

# Example 2: Predict for January 2026
test_auction_month = pd.Timestamp('2026-01')
test_market_month = pd.Timestamp('2026-01')

# The code will automatically find and predict ALL outage dates in that month
```

**Change training period in Cell 1 (STEP 1):**

```python
# Example: Use 12 months of training data
train_start = pd.Timestamp('2024-10-01')
train_end = pd.Timestamp('2025-09-30')
```

Then run all cells sequentially!

---

In [49]:
sf = pd.read_parquet('/opt/temp/tmp/pw_data/spice6/prod_f0p_model_miso/sf/auction_month=2025-10/market_month=2025-10/market_round=1/outage_date=2025-10-01/sf.parquet')
score = pd.read_parquet('/opt/temp/tmp/pw_data/spice6/prod_f0p_model_miso/density/auction_month=2025-10/market_month=2025-10/market_round=1/outage_date=2025-10-01/score.parquet')

cons = pd.read_parquet(f'/opt/temp/tmp/pw_data/spice6/prod_f0p_model_miso/constraint_info/auction_month=2025-10/market_round=1/period_type=f0/class_type=onpeak/constraints.parquet')
spice_map = cons[cons['convention'] != 999].dropna(subset=['branch_name']).groupby('constraint_id')['branch_name'].apply(','.join)

In [51]:
useful_score = score[score['score'] > 0.001]
useful_score.shape, score.shape, len(useful_score.index.get_level_values(0).unique())

((3960, 1), (25292, 1), 3668)

In [52]:
ssf = sf[useful_score.index.get_level_values(0).unique()]
ssf.columns = ssf.columns.map(spice_map)

In [57]:
C = ssf.T.groupby(level=0).mean().T.corr()

In [66]:
C.loc[:, C.columns.str.contains('AUST_TAYS_1545', case=False)].squeeze().sort_values().dropna()

POSTROCK XF1       -0.534861
LACYGN_STILWE1 1   -0.525606
NEBRCSUB3434_1 1   -0.509485
NBOROORENT34_1 1   -0.506624
HOYTJEFF34_1   1   -0.499416
                      ...   
ROOT_MTGY_5218 A    0.586264
OLI-UNPA       1    0.590253
112_WILT CT_P       0.609858
112_WILT X001       0.609858
AUST_TAYS_1545 A    1.000000
Name: AUST_TAYS_1545 A, Length: 1248, dtype: float64

In [ ]:
# ============================================================================
# SAMPLE CODE: Extend Binding Events Using Shift Factor Correlation
# ============================================================================
# This cell demonstrates how to augment training data for branches with 
# insufficient binding events by borrowing samples from highly correlated branches

print("\n[SAMPLE] Shift Factor-Based Training Data Augmentation")
print("=" * 80)

# ============================================================================
# STEP 1: Load Shift Factor Data
# ============================================================================

print("\n[1] Loading Shift Factor Data...")

sf_path = Path('/opt/temp/tmp/pw_data/spice6/prod_f0p_model_miso/sf/sf.parquet')

if sf_path.exists():
    sf_data = pd.read_parquet(sf_path)
    print(f"✓ Shift factor data loaded: {sf_data.shape}")
    print(f"  Columns: {list(sf_data.columns)}")
    print(f"\nFirst few rows:")
    print(sf_data.head())
else:
    print(f"⚠️  Shift factor file not found at: {sf_path}")
    sf_data = None

# ============================================================================
# STEP 2: Build Correlation Matrix from Shift Factors
# ============================================================================

if sf_data is not None:
    print("\n[2] Building Branch Correlation Matrix...")
    
    # Assuming sf_data has structure like:
    # - 'branch_name_1', 'branch_name_2', 'correlation' (correlation coefficient)
    # OR
    # - Rows are branches, columns are branches, values are correlations
    
    # Example: If data is in long format (branch_name_1, branch_name_2, correlation)
    # Convert to correlation matrix
    
    # Method 1: If already a correlation matrix
    if 'branch_name_1' in sf_data.columns and 'branch_name_2' in sf_data.columns:
        # Long format - create pivot table
        print("  Format: Long format (branch_name_1, branch_name_2, correlation)")
        
        # Build symmetric correlation matrix
        correlation_matrix = sf_data.pivot_table(
            index='branch_name_1',
            columns='branch_name_2',
            values='correlation',  # or 'shift_factor' depending on column name
            fill_value=0
        )
        print(f"  Correlation matrix shape: {correlation_matrix.shape}")
        
    else:
        # Assume it's already a matrix format
        print("  Format: Matrix format (branches × branches)")
        correlation_matrix = sf_data
        print(f"  Correlation matrix shape: {correlation_matrix.shape}")
    
    print(f"\n  Sample correlations:")
    print(correlation_matrix.iloc[:5, :5])

# ============================================================================
# STEP 3: Find Correlated Branches for Each Branch
# ============================================================================

if sf_data is not None:
    print("\n[3] Finding Highly Correlated Branches (correlation > 0.9)...")
    
    CORRELATION_THRESHOLD = 0.9
    
    # Dictionary to store correlated branches
    correlated_branches = {}  # {branch_name: [list of correlated branch names]}
    
    for branch_name in correlation_matrix.index:
        # Get correlations for this branch
        correlations = correlation_matrix.loc[branch_name]
        
        # Find branches with correlation > threshold (excluding self)
        highly_correlated = correlations[
            (correlations > CORRELATION_THRESHOLD) & 
            (correlations.index != branch_name)
        ]
        
        if len(highly_correlated) > 0:
            correlated_branches[branch_name] = highly_correlated.index.tolist()
    
    print(f"✓ Found {len(correlated_branches):,} branches with correlated counterparts")
    
    # Show examples
    print(f"\n  Examples of correlated branches:")
    example_count = 0
    for branch, correlated in correlated_branches.items():
        if len(correlated) > 0 and example_count < 5:
            print(f"    {branch[:50]}: {len(correlated)} correlated branches")
            for corr_branch in correlated[:3]:
                corr_value = correlation_matrix.loc[branch, corr_branch]
                print(f"      → {corr_branch[:45]} (correlation: {corr_value:.3f})")
            if len(correlated) > 3:
                print(f"      ... and {len(correlated) - 3} more")
            example_count += 1

# ============================================================================
# STEP 4: Augment Training Data for Branches with Insufficient Binding Events
# ============================================================================

if sf_data is not None and 'train_data_by_branch' in locals():
    print("\n[4] Augmenting Training Data Using Correlated Branches...")
    
    MIN_BINDING_REQUIRED = 5  # Minimum binding samples to train regressor
    
    # Track augmentation statistics
    augmentation_stats = []
    
    for branch_name, branch_data in train_data_by_branch.items():
        # Count binding events for this branch
        n_binding = (branch_data['label'] > 0).sum()
        
        # Only augment if insufficient binding events
        if n_binding < MIN_BINDING_REQUIRED:
            
            # Check if this branch has correlated branches
            if branch_name in correlated_branches:
                correlated_list = correlated_branches[branch_name]
                
                # Collect binding samples from correlated branches
                augmented_samples = []
                
                for corr_branch in correlated_list:
                    # Check if correlated branch exists in training data
                    if corr_branch in train_data_by_branch:
                        corr_data = train_data_by_branch[corr_branch]
                        
                        # Get only binding samples from correlated branch
                        binding_samples = corr_data[corr_data['label'] > 0]
                        
                        if len(binding_samples) > 0:
                            augmented_samples.append(binding_samples)
                
                # Combine augmented samples
                if len(augmented_samples) > 0:
                    augmented_data = pd.concat(augmented_samples, axis=0)
                    original_binding = n_binding
                    augmented_binding = len(augmented_data)
                    
                    augmentation_stats.append({
                        'branch_name': branch_name,
                        'original_binding': original_binding,
                        'augmented_binding': augmented_binding,
                        'correlated_sources': len(augmented_samples),
                        'total_after_augmentation': original_binding + augmented_binding
                    })
    
    # Show augmentation results
    if len(augmentation_stats) > 0:
        aug_df = pd.DataFrame(augmentation_stats)
        
        print(f"\n✓ Augmentation Summary:")
        print(f"  Branches augmented: {len(aug_df):,}")
        print(f"  Total binding samples added: {aug_df['augmented_binding'].sum():,}")
        print(f"  Average augmentation per branch: {aug_df['augmented_binding'].mean():.1f} samples")
        
        # Show top augmented branches
        print(f"\n  Top 10 Branches by Augmentation:")
        top_augmented = aug_df.nlargest(10, 'augmented_binding')
        for idx, row in top_augmented.iterrows():
            print(f"    {row['branch_name'][:45]:45s}: "
                  f"{row['original_binding']:2d} → {row['total_after_augmentation']:3d} "
                  f"(+{row['augmented_binding']:3d} from {row['correlated_sources']} sources)")
        
        # Show distribution
        print(f"\n  Augmentation Distribution:")
        print(f"    Branches with 0 original → 5+ after: {(aug_df['total_after_augmentation'] >= 5).sum():,}")
        print(f"    Branches still <5 after augmentation: {(aug_df['total_after_augmentation'] < 5).sum():,}")
    else:
        print(f"\n  No branches qualified for augmentation")
        print(f"  (Either all have sufficient data or no correlations found)")

# ============================================================================
# STEP 5: Example Usage - Train Model with Augmented Data
# ============================================================================

print("\n[5] Example: Training with Augmented Data")
print("-" * 80)

if sf_data is not None and len(augmentation_stats) > 0:
    # Show example for one augmented branch
    example_branch = aug_df.iloc[0]['branch_name']
    
    print(f"\nExample Branch: {example_branch}")
    
    # Original data
    original_data = train_data_by_branch[example_branch]
    original_binding = (original_data['label'] > 0).sum()
    
    print(f"  Original binding samples: {original_binding}")
    
    # Get augmented samples from correlated branches
    if example_branch in correlated_branches:
        correlated_list = correlated_branches[example_branch][:3]  # Show first 3
        
        print(f"  Correlated branches used for augmentation:")
        for corr_branch in correlated_list:
            if corr_branch in train_data_by_branch:
                corr_data = train_data_by_branch[corr_branch]
                corr_binding = (corr_data['label'] > 0).sum()
                corr_value = correlation_matrix.loc[example_branch, corr_branch]
                print(f"    • {corr_branch[:40]:40s}: {corr_binding:3d} binding samples (corr={corr_value:.3f})")
        
        print(f"\n  Training Strategy:")
        print(f"    1. Combine original data with augmented samples from correlated branches")
        print(f"    2. Train regressor on combined dataset")
        print(f"    3. Model benefits from {aug_df.iloc[0]['augmented_binding']} additional binding samples")
else:
    print("  Run previous steps to see augmentation example")

print("\n" + "=" * 80)
print("NOTE: This is SAMPLE CODE demonstrating the concept.")
print("To integrate into the main workflow:")
print("  1. Run this cell to verify shift factor data loads correctly")
print("  2. Check correlation matrix structure matches expectations")
print("  3. Integrate augmentation logic into STEP 6 (model training)")
print("  4. Track which models used augmented data vs. original only")
print("=" * 80)